# HuggingFace Trainer Pipeline

Base model: MARBERTv2

**Setup**

In [ ]:
import os
from google.colab import userdata
!git config --global user.name "sivrox"
repo_url = "https://github.com/sivrox/Arabic-English-Sentiment-Analysis-Project.git"

if not os.path.exists("/content/Arabic-English-Sentiment-Analysis-Project"):
    os.system(f"git clone {repo_url}")
else:
    os.system("cd /content/Arabic-English-Sentiment-Analysis-Project && git pull")

%cd /content/Arabic-English-Sentiment-Analysis-Project
print("Repository cloned.")
!ls

In [ ]:
# install required packages
!pip install -q transformers==4.39.0 peft==0.10.0 accelerate==0.28.0 scikit-learn evaluate seaborn matplotlib nltk pyarrow pandas datasets emoji

In [ ]:
import os, sys, random, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import evaluate
import nltk
nltk.download("punkt", quiet=True)

# fix random seed for reproducibility — must match Notebook 01 for a fair comparison
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# add repo root to path so we can import our preprocessing module
sys.path.insert(0, "/content/Arabic-English-Sentiment-Analysis-Project")

# ---
# config — identical to Notebook 01 for a fair cross-framework comparison
# batch_size=32 and epochs=10 take advantage of A100's 40GB VRAM
# ---
model_name = "UBC-NLP/MARBERTv2"
max_len    = 128
batch_size = 16
epochs     = 3
lr_full    = 2e-5
lr_lora    = 3e-4
num_labels = 3
patience   = 3

# label encoding — must match Notebook 01
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

# create output folder for plots and CSVs
Path("results/huggingface").mkdir(parents=True, exist_ok=True)
print("Config ready.")

**Load Data**

In [ ]:
from preprocessing.preprocessor import build_hf_datasets

data_path = "preprocessing/datasets/processed/unified_raw.csv"

# build_hf_datasets runs the same cleaning pipeline as build_dataloaders in Notebook 01
# but returns HuggingFace Dataset objects instead of PyTorch DataLoaders
# the Trainer API requires HuggingFace Dataset format — it cannot accept torch DataLoaders
print("Building HuggingFace datasets for MARBERT...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_ds, val_ds, test_ds = build_hf_datasets(
    data_path, tokenizer, max_length=max_len, seed=seed,
    save_cleaned_csv=False  # already saved by Notebook 01
)

print(f"\nDatasets ready — Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

**Helper Functions**

In [ ]:
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    # called automatically by the Trainer after each evaluation epoch
    # returns a dict — the Trainer logs these values and uses macro_f1 for early stopping
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_score(labels, preds)
    return {"macro_f1": macro_f1, "accuracy": acc}

def plot_trainer_history(trainer, title, save_path):
    # extract training log from the Trainer's internal state object
    # Trainer logs at steps rather than epochs so x-axis is eval step count
    log = trainer.state.log_history
    train_loss = [e["loss"] for e in log if "loss" in e and "eval_loss" not in e]
    val_loss = [e["eval_loss"] for e in log if "eval_loss" in e]
    val_f1 = [e["eval_macro_f1"] for e in log if "eval_macro_f1" in e]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    if train_loss: ax1.plot(train_loss, label="Train")
    if val_loss: ax1.plot(val_loss, label="Val")
    ax1.set(title=f"{title} — Loss", xlabel="Step", ylabel="Loss")
    ax1.legend()
    if val_f1: ax2.plot(val_f1, label="Val Macro F1", color="green")
    ax2.set(title=f"{title} — Macro F1", xlabel="Eval Step", ylabel="F1")
    ax2.legend()
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()

def plot_confusion_matrix(preds, labels, title, save_path):
    # shows per-class prediction breakdown
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Negative", "Neutral", "Positive"],
                yticklabels=["Negative", "Neutral", "Positive"])
    plt.title(f"{title} — Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()

def print_report(preds, labels, title):
    # per-class precision, recall, F1 from sklearn
    print(f"\n{title} — Classification Report")
    print(classification_report(labels, preds, target_names=["Negative", "Neutral", "Positive"]))

def full_evaluation(trainer, dataset, title, cm_save_path):
    # run trainer.predict() on the test set and compute all final metrics
    # test set is never used during training or early stopping — only here at the end
    preds_out = trainer.predict(dataset)
    preds = np.argmax(preds_out.predictions, axis=-1)
    labels = preds_out.label_ids
    macro_f1 = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    print(f"\n{title} — Test Macro F1: {macro_f1:.4f}  |  Accuracy: {acc:.4f}")
    print_report(preds, labels, title)
    plot_confusion_matrix(preds, labels, title, cm_save_path)
    return macro_f1, acc, preds, labels

def make_args(output_dir, lr):
    # shared TrainingArguments used by both strategies — only output_dir and lr differ
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval="epoch",       # evaluate on val set after every epoch
        save_strategy="epoch",             # save checkpoint after every epoch
        load_best_model_at_end=True,       # automatically restore best checkpoint when training ends
        metric_for_best_model="macro_f1",  # use macro F1 to decide which checkpoint is best
        greater_is_better=True,
        logging_steps=100,
        seed=seed,
        fp16=torch.cuda.is_available(),    # mixed precision for faster A100 training
        report_to="none",                  # disable wandb/tensorboard logging
    )

Strategy A - **Full Fine-Tuning**

In [ ]:
# load pretrained MARBERT with a fresh 3-class classification head
# the Trainer handles the optimizer, scheduler, gradient clipping, and checkpointing
print("Loading MARBERT for Full Fine-Tuning...")
model_fft = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True  # suppresses expected warning about new head weights
)
trainable = sum(p.numel() for p in model_fft.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_fft.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

trainer_fft = Trainer(
    model=model_fft,
    args=make_args("hf_outputs/marbert_fft", lr_full),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)],
)
print("\nTraining MARBERT Full Fine-Tuning...")
trainer_fft.train()
print("Done.")

In [ ]:
# plot training curves then evaluate on held-out test set
plot_trainer_history(trainer_fft, "MARBERT Full FT (HF)", "results/huggingface/marbert_fft_curves.png")
f1_fft, acc_fft, preds_fft, labels_test = full_evaluation(trainer_fft, test_ds, "MARBERT Full FT (HF)", "results/huggingface/marbert_fft_cm.png")

Strategy B - **LoRA (peft library)**

In [ ]:
# this notebook uses the HuggingFace peft library LoraConfig
# Notebook 01 uses the from-scratch implementation in peft_implementation.py
# same rank (r=8), alpha (16), dropout (0.1), and target layers — results are directly comparable
print("Loading MARBERT for LoRA...")
lora_base = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True
)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],  # same layers targeted by peft_implementation.py
    bias="none",
)
model_lora = get_peft_model(lora_base, lora_config)
# confirms expected reduction: ~1M trainable out of 163M total parameters
model_lora.print_trainable_parameters()

trainer_lora = Trainer(
    model=model_lora,
    args=make_args("hf_outputs/marbert_lora", lr_lora),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)],
)
print("\nTraining MARBERT LoRA...")
trainer_lora.train()
print("Done.")

In [ ]:
plot_trainer_history(trainer_lora, "MARBERT LoRA (HF)", "results/huggingface/marbert_lora_curves.png")
f1_lora, acc_lora, preds_lora, _ = full_evaluation(trainer_lora, test_ds, "MARBERT LoRA (HF)", "results/huggingface/marbert_lora_cm.png")

**Cross-Framework Comparison**

In [ ]:
#save HuggingFace results
hf_results = pd.DataFrame({
    "model" : ["MARBERT", "MARBERT"],
    "strategy" : ["Full FT", "LoRA"],
    "framework" : ["HF Trainer", "HF Trainer"],
    "test_macro_f1" : [round(f1_fft, 4), round(f1_lora,  4)],
    "test_accuracy" : [round(acc_fft, 4), round(acc_lora, 4)],
})
hf_results.to_csv("results/huggingface/hf_comparison.csv", index=False)
print("HuggingFace results:")
print(hf_results.to_string(index=False))

#load PyTorch results from pytorch_training_pipeline
pytorch_csv = Path("results/pytorch/pytorch_comparison.csv")
if pytorch_csv.exists():
    pt_results = pd.read_csv(pytorch_csv)
    print(f"\nPyTorch results loaded from {pytorch_csv}:")
    print(pt_results.to_string(index=False))
else:
    print(f"\nWarning: {pytorch_csv} not found. Run Notebook 01 first then re-run this cell.")
    pt_results = None

In [ ]:
if pt_results is not None:
    cross = pd.concat([
        pt_results[["model", "strategy", "framework", "test_macro_f1", "test_accuracy"]], #combine both strategy metrics into one comparison table
        hf_results[["model", "strategy", "framework", "test_macro_f1", "test_accuracy"]],
    ], ignore_index=True)
    cross.to_csv("results/cross_framework_comparison.csv", index=False)
    print("Cross-framework comparison:")
    print(cross.to_string(index=False))

    # grouped bar chart — one group per strategy, one bar per framework
    labels_x = cross[cross["framework"] == "PyTorch"]["strategy"].tolist()
    pt_f1 = cross[cross["framework"] == "PyTorch"]["test_macro_f1"].tolist()
    hf_f1 = cross[cross["framework"] == "HF Trainer"]["test_macro_f1"].tolist()

    x = np.arange(len(labels_x))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    bars1 = ax.bar(x - width/2, pt_f1, width, label="PyTorch (Notebook 01)", color="#4C72B0", alpha=0.85)
    bars2 = ax.bar(x + width/2, hf_f1, width, label="HF Trainer (Notebook 02)", color="#DD8452", alpha=0.85)
    ax.bar_label(bars1, fmt="%.4f", padding=3, fontsize=9)
    ax.bar_label(bars2, fmt="%.4f", padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_x)
    ax.set_ylim(0.7, 1.0)
    ax.set_ylabel("Test Macro F1")
    ax.set_title("Cross-Framework Comparison — MARBERT: PyTorch vs HuggingFace Trainer")
    ax.legend()
    plt.tight_layout()
    plt.savefig("results/cross_framework_chart.png", dpi=150)
    plt.show()
    plt.close()
    print("\nSaved: results/cross_framework_comparison.csv and results/cross_framework_chart.png")
else:
    print("Run Notebook 01 first to generate the PyTorch results CSV.")

**Save to GitHub**

In [ ]:
#push all results to github
os.system("git add results/ -f")
os.system('git commit -m "Add HuggingFace training results and cross-framework comparison"')
os.system("git push origin main")
print("Pushed Results to GitHub.")

**Save models to Google Drive**

In [ ]:
#Save HuggingFace Trainer checkpoints to Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount("/content/drive")
save_dir = "/content/drive/MyDrive/sentimentgulf_checkpoints"
os.makedirs(save_dir, exist_ok=True)
print("Saving HuggingFace Trainer checkpoints to Drive...")

#Full FT Checkpoints
trainer_fft.save_model(f"{save_dir}/hf_marbert_fft")
tokenizer.save_pretrained(f"{save_dir}/hf_marbert_fft")
print(f"Saved: hf_marbert_fft")

#LoRA Checkpoints
trainer_lora.save_model(f"{save_dir}/hf_marbert_lora")
tokenizer.save_pretrained(f"{save_dir}/hf_marbert_lora")
print(f"Saved: hf_marbert_lora")

print(f"\nAll checkpoints saved to: {save_dir}")
print("Note: hf_marbert_fft is a full model checkpoint (~600MB).")
print("hf_marbert_lora contains only the LoRA adapter weights (much smaller).")